<a href="https://colab.research.google.com/github/emmakelly7/ds2002-fa26/blob/main/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    last = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'not retryable: {r.status_code} {r.text[:200]}')

            last = f'status {r.status_code}'

        except requests.RequestException as e:
            last = repr(e)

        if attempt < tries - 1:
            delay = 0.5 * (2 ** attempt)
            print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
            time.sleep(delay)

    raise RuntimeError(f'gave up after {tries} attempts: {last}')

I used requests.get() with a timeout of 15-seconds, so the program would wait for a specific time for a response. If there is a temporary error, like the 429, then it will retry up to three times and increases the delay between them. The except requests.RequestException is for errors and makes sure that the errors you cannot retry with will raise an error message. If the request works, then the daily JSON is converted into a pandas dataframe. However, if it fails, then there will be an error.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [11]:
used_fallback = False

try:
    wx = get_weather('2026-09-05', '2026-10-03')
    wx = wx[wx['time'].isin(['2026-09-05', '2026-10-03'])].copy()

    if len(wx) != 2:
        raise ValueError(f'expected 2 dates, got {len(wx)}')

    print(wx)

except Exception as e:
    print('Weather request failed:', e)
    used_fallback = True

          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
28  2026-10-03                22.0               14.7


I used the get_weather() from to get the weather data for September 5 and October 3, 2026. I used .isin() to make sure the dataframe only had the two specific dates. Then, I used the try and except in case there was a to failed API request. If it fails, then the code would record it and change used_fallback to True. This helps to discern the real and fallback data.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'

print(wx)


          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
28  2026-10-03                22.0               14.7  open-meteo


I used if/else to check if the API request from Part 2 worked. If it had failed, it would have output as "fallback sample" because it would have used a copy of the fallback data. However, it output the source as open-meteo, so I know it was successful and used the actual data. Additionally, I added a column for source, so it is more clear where the data was coming from.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

print(wx)

         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo


In [5]:
print(wx.dtypes)

date         datetime64[ns]
temp_max            float64
precip_mm           float64
source               object
dtype: object


I cleaned the data from the weather DataFrame to make it easier to join with other data. I used .rename() to change the original column names to date, temp_max, and precip_mm, but I kept the source column. Then, I used pd.to_datetime() to change the date values from strings into actual values. Then, I did wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
to put the four columns in the right order.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [6]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


I uncommented the assertions and made them pass, so I was able to understand if it was ready to be joined with other data. The first assertion is checking if the four columns have the right names and are ordered correctly. The second assertion checks if there are the two rows. The third is checking to see if the data column is a string or datetime. The fourth and fifth assertions are making sure there are no missing precipitation values and that none of them are negative. It output that it was safe to join, so we know it can be used properly now.

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

difference = wetter['precip_mm'] - drier['precip_mm']

print(f"{wetter['date'].strftime('%B %d, %Y')} was wetter than {drier['date'].strftime('%B %d, %Y')} by {difference:.1f} mm of precipitation.")

October 03, 2026 was wetter than September 05, 2026 by 12.2 mm of precipitation.


.idxmax() and .idxmin() output which date had the most and least precipitation. .loc[] selects those rows and then subtracts the precipitation amounts to find the difference. Then, I used an f-string to print a sentence that a vendor could read to help explain which date was wetter and by how much. The output states that October 3, 2026 was wetter than September 5, 2026 by 12.2 mm.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
from datetime import datetime

print('API endpoint:', BASE)
print('Coordinates: 38.03, -78.51')
print('Date range: 2026-09-05 to 2026-10-03')
print('Retrieved on:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Data source:', 'fallback sample' if used_fallback else 'open-meteo')

API endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Retrieved on: 2026-10-08 22:38:06
Data source: open-meteo


I printed the API endpoint, coordinates, date range, when I pulled it, and the data source to output where the weather data came from. I used  BASE from Part 1 to use the API endpoint and datetime.now() to find the time. I also used used_fallback to show where the data came from, whether Open-Meteo or the fallback data. My API request was successful, so the source output "open-meteo."

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure instead of raising, I could have continued the assignment without knowing that it had failed. If I merged it onto 200 days of sales and used a left join, the data from the sales would be there, but the precipitation column would be missing data. If I then charted rain against revenue, the chart might not show data points or it could have misleading information. I might not catch it because it did not raise an error, and the sales data would look like it was there. Therefore, it is very important to raise an error if an API request fails, and you have to be careful to validate the data.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
# stretch
orlando = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orlando = orlando[orlando['time'].isin(['2026-09-05', '2026-10-03'])].copy()
orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

orlando['date'] = pd.to_datetime(orlando['date'])

wx['city'] = 'Charlottesville'
orlando['city'] = 'Orlando'
orlando['source'] = 'open-meteo'

both_cities = pd.concat([wx, orlando], ignore_index=True)

print(both_cities)

        date  temp_max  precip_mm      source             city
0 2026-09-05      29.4        2.5  open-meteo  Charlottesville
1 2026-10-03      22.0       14.7  open-meteo  Charlottesville
2 2026-09-05      32.1        3.9  open-meteo          Orlando
3 2026-10-03      30.4       14.5  open-meteo          Orlando


In [10]:
print('Number of rows:', len(both_cities))
print('Cities:', both_cities['city'].unique())

Number of rows: 4
Cities: ['Charlottesville' 'Orlando']
